# InternVL2.5-4B Image Phrase Extractor — Google Colab (Google Drive)

**Model:** `OpenGVLab/InternVL2_5-4B` — publicly available, no HuggingFace login required.

**Output per image (saved to `phrases_internvl.csv`):**

| filename | phrase | meaningful_tokens | word_count | meaningful_token_count |
|---|---|---|---|---|
- `word_count` — total words in the phrase
- `meaningful_token_count` — words after removing stopwords (a, the, is, in, of, ...)

### Before running:

1. Go to [drive.google.com](https://drive.google.com) and upload your image folder into **My Drive**
2. Set `DRIVE_IMAGE_FOLDER` in Cell 1 to the exact folder name you uploaded
3. **Allow popups** for `colab.research.google.com` — the Drive mount opens a Google sign-in popup

**If Cell 1 fails with a credential error:**
- Allow popups in your browser settings and re-run Cell 1
- Or try: `Runtime → Restart session` then re-run

**Runtime recommendation:** `Runtime → Change runtime type → T4 GPU` — model is ~8 GB in bfloat16.

In [ ]:
# ── Cell 1: Mount Google Drive and set image folder ───────────────────────────
from google.colab import drive
import os, zipfile

drive.mount('/content/drive', force_remount=True)

# ↓ Change this to the name of the folder you uploaded to My Drive
DRIVE_IMAGE_FOLDER = 'vqaimages'

DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_IMAGE_FOLDER}'

assert os.path.isdir(DRIVE_DIR), (
    f'Folder not found: {DRIVE_DIR}\n'
    f'Make sure "{DRIVE_IMAGE_FOLDER}" exists inside My Drive on Google Drive.'
)

# Extract ZIPs if present
zip_files = [f for f in os.listdir(DRIVE_DIR) if f.lower().endswith('.zip')]
if zip_files:
    EXTRACT_DIR = f'/content/{DRIVE_IMAGE_FOLDER}_extracted'
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    for zf in zip_files:
        print(f'Extracting {zf} ...')
        with zipfile.ZipFile(os.path.join(DRIVE_DIR, zf), 'r') as z:
            z.extractall(EXTRACT_DIR)
    IMAGE_DIR = EXTRACT_DIR
    print(f'Extracted to {IMAGE_DIR}')
else:
    IMAGE_DIR = DRIVE_DIR
    print(f'Reading images directly from {IMAGE_DIR}')

total = sum(1 for _, _, fs in os.walk(IMAGE_DIR) for f in fs)
print(f'Found {total} file(s) in total')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
# transformers is pinned to 4.45.2 — InternVL2.5 is incompatible with 4.46+
# which added mark_tied_weights_as_initialized / all_tied_weights_keys.
!pip install -q "transformers==4.45.2" accelerate einops timm Pillow opencv-python-headless numpy torch torchvision

# ⚠️ After this cell finishes, go to Runtime → Restart session,
# then run all cells again from Cell 1.
print('Done. Now go to Runtime → Restart session, then re-run all cells.')

In [ ]:
# ── Cell 3a: Patch cached source + clear module cache ────────────────────────
# The crash occurs because:
#   1. torch.linspace(...).item() is called during model __init__
#   2. transformers wraps __init__ in init_empty_weights() which makes ALL
#      tensors (including linspace output) into meta tensors
#   3. .item() on meta tensors raises RuntimeError
#
# Fix A: Replace torch.linspace with pure-Python math in the cached file
# Fix B: Clear Python's module cache so the patched file is actually used

import pathlib, sys

CACHE_ROOT = pathlib.Path.home() / '.cache/huggingface/modules/transformers_modules/OpenGVLab'

# ── Fix A: patch the file ─────────────────────────────────────────────────────
BAD_LINE  = 'dpr = [x.item() for x in torch.linspace(0, config.drop_path_rate, config.num_hidden_layers)]'
GOOD_LINE = ('_n   = int(config.num_hidden_layers)\n'
             '        _end = config.drop_path_rate\n'
             '        _end = _end if isinstance(_end, float) else 0.1\n'
             '        dpr  = [_end * i / max(_n - 1, 1) for i in range(_n)]')

patched = 0
for vit_file in CACHE_ROOT.rglob('modeling_intern_vit.py'):
    code = vit_file.read_text(encoding='utf-8')
    if BAD_LINE in code:
        vit_file.write_text(code.replace(BAD_LINE, GOOD_LINE), encoding='utf-8')
        print(f'Patched file: {vit_file}')
        patched += 1
    elif '_end = config.drop_path_rate' in code:
        print(f'Already patched: {vit_file}')
        patched += 1

if patched == 0:
    print('WARNING: file not found — check CACHE_ROOT path')

# ── Fix B: evict cached module so Python reloads the patched file ─────────────
evicted = [k for k in list(sys.modules) if 'internvl' in k.lower() or 'intern_vit' in k.lower()]
for k in evicted:
    del sys.modules[k]
print(f'Cleared {len(evicted)} cached module(s): {evicted}')
print('Ready — run Cell 4 now.')

In [ ]:
# ── Cell 4: Load InternVL2.5-4B model ────────────────────────────────────────
# Must run Cell 3a before this cell.
# low_cpu_mem_usage=False avoids init_empty_weights() entirely — no meta tensors.
# Model is initialised on CPU then moved to GPU in one step.
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_ID = 'OpenGVLab/InternVL2_5-4B'
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE    = torch.bfloat16 if DEVICE == 'cuda' else torch.float32

print(f'Loading {MODEL_ID} on {DEVICE.upper()} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)
model     = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    trust_remote_code=True,
    low_cpu_mem_usage=False,   # disables init_empty_weights() — no meta tensors
).eval().to(DEVICE)

print(f'Model ready  |  device: {DEVICE.upper()}  |  dtype: {DTYPE}')

In [ ]:
# ── Cell 5: Configuration ─────────────────────────────────────────────────────
MAX_NEW_TOKENS   = 40
FRAMES_PER_VIDEO = 4
IMAGE_SIZE       = 448   # InternVL2 native tile size
MAX_TILES        = 6     # max dynamic tiles per image

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.wmv', '.m4v'}

PROMPT = (
    'Describe this image in one short phrase focusing on '
    'whether fire is present, its nature (e.g. wildfire, campfire, flame, explosion, etc), '
    'and the surrounding environment. '
    'Reply with the phrase only.'
)

# Common English stop words to filter out when computing meaningful tokens
STOPWORDS = {
    'a','an','the','is','are','was','were','be','been','being',
    'in','on','at','to','of','for','with','by','from','into',
    'and','or','but','not','no','as','its','it','this','that',
    'there','their','they','has','have','had','do','does','did',
    'i','we','you','he','she','over','under','near','between',
    'through','around','against','during','without','within', 'setting', 'environment'
}

In [ ]:
# ── Cell 6: Helper functions ──────────────────────────────────────────────────
import time
from pathlib import Path

import cv2
import numpy as np
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode

# ── Image preprocessing (InternVL2 dynamic tiling) ───────────────────────────
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def build_transform(size=IMAGE_SIZE):
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB')),
        T.Resize((size, size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def dynamic_preprocess(image, min_num=1, max_num=MAX_TILES, image_size=IMAGE_SIZE):
    w, h    = image.size
    aspect  = w / h
    ratios  = sorted(
        {(i, j) for n in range(min_num, max_num + 1)
                for i in range(1, n + 1)
                for j in range(1, n + 1)
                if min_num <= i * j <= max_num},
        key=lambda x: x[0] * x[1]
    )
    best = min(ratios, key=lambda r: abs(aspect - r[0] / r[1]))
    tw, th  = image_size * best[0], image_size * best[1]
    resized = image.resize((tw, th))
    tiles   = [
        resized.crop((
            (idx % best[0]) * image_size, (idx // best[0]) * image_size,
            ((idx % best[0]) + 1) * image_size, ((idx // best[0]) + 1) * image_size,
        ))
        for idx in range(best[0] * best[1])
    ]
    tiles.append(image.resize((image_size, image_size)))  # thumbnail
    return tiles

def load_pixel_values(image):
    transform = build_transform()
    tiles     = dynamic_preprocess(image.convert('RGB'))
    return torch.stack([transform(t) for t in tiles]).to(DTYPE).to(DEVICE)

# ── Inference ─────────────────────────────────────────────────────────────────

def get_phrase(image):
    pixel_values = load_pixel_values(image)
    gen_cfg      = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    response     = model.chat(tokenizer, pixel_values, PROMPT, gen_cfg)
    return response.strip()

# ── Text analysis ──────────────────────────────────────────────────────────────

def analyse_phrase(phrase):
    words             = phrase.split()
    meaningful_tokens = [w.strip('.,;:!?\'"').lower()
                         for w in words
                         if w.strip('.,;:!?\'"').lower() not in STOPWORDS
                         and w.strip('.,;:!?\'"')]
    return {
        'word_count'        : len(words),
        'meaningful_tokens' : meaningful_tokens,
        'token_count'       : len(meaningful_tokens),  # count of meaningful words
    }

# ── Video frame extraction ─────────────────────────────────────────────────────

def extract_frames(video_path, n=FRAMES_PER_VIDEO):
    cap   = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        return []
    frames = []
    for idx in np.linspace(0, total - 1, n, dtype=int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if ok:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()
    return frames

print('Helper functions ready.')

In [ ]:
# ── Cell 7: Process all images / videos ───────────────────────────────────────
import csv

root     = Path(IMAGE_DIR)
out_path = Path('/content/drive/MyDrive/phrases_internvl.csv')

all_files = sorted(
    p for p in root.rglob('*')
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS | VIDEO_EXTS
)

print(f'Found {len(all_files)} file(s)\n' + '─' * 60)

COLUMNS = ['filename', 'phrase', 'meaningful_tokens', 'word_count', 'meaningful_token_count']

with open(out_path, 'w', encoding='utf-8-sig', newline='') as f:
    # utf-8-sig writes a BOM so Excel opens it correctly without encoding issues
    writer = csv.DictWriter(f, fieldnames=COLUMNS)
    writer.writeheader()

    for idx, path in enumerate(all_files, 1):
        rel = path.relative_to(root)
        ext = path.suffix.lower()
        print(f'[{idx}/{len(all_files)}] {rel}')

        def write_row(filename, phrase, frame_label=''):
            info  = analyse_phrase(phrase)
            label = str(filename) + (f' {frame_label}' if frame_label else '')
            writer.writerow({
                'filename'               : label,
                'phrase'                 : phrase,
                'meaningful_tokens'      : ' '.join(info['meaningful_tokens']),
                'word_count'             : info['word_count'],
                'meaningful_token_count' : info['token_count'],
            })
            f.flush()
            print(f'  phrase      : {phrase!r}')
            print(f'  meaningful  : {info["meaningful_tokens"]}')
            print(f'  words / meaningful: {info["word_count"]} / {info["token_count"]}')

        try:
            t0 = time.time()

            if ext in IMAGE_EXTS:
                phrase = get_phrase(Image.open(path))
                write_row(rel, phrase)
                print(f'  elapsed     : {time.time()-t0:.1f}s')

            else:  # video
                frames = extract_frames(path)
                for i, frame in enumerate(frames):
                    phrase = get_phrase(frame)
                    write_row(rel, phrase, frame_label=f'[frame {i+1}]')
                    print(f'  frame {i+1}/{len(frames)} elapsed: {time.time()-t0:.1f}s')
                    t0 = time.time()

        except Exception as e:
            print(f'  ERROR: {e}')
            writer.writerow({
                'filename': str(rel), 'phrase': f'ERROR: {e}',
                'meaningful_tokens': '', 'word_count': '', 'meaningful_token_count': ''
            })
            f.flush()

print(f'\nDone. phrases_internvl.csv saved to My Drive')